In [ ]:
### Step 1 — Drop CO2 and irrelevant columns

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/processed/routes_processed.csv')
print(df.shape)
df.head()

In [ ]:
print(df.dtypes)
print(df['is_underserved'].value_counts(normalize=True))

In [ ]:
# Drop columns that are useless for is_underserved prediction
cols_to_drop = [
    # Duplicate column
    'type',
    # CO2 columns - from old project, no predictive value for underserved
    'train_gco2_pkm',
    'plane_gco2_pkm',
    'train_co2_kg',
    'plane_co2_kg',
    'co2_savings_kg',
    'savings_percent',
    'emission_source',
    'co2_ratio_train_plane',
    'train_co2_per_passenger',
    'plane_co2_per_passenger',
    # Date column - not useful for classification
    'calculation_date',
]

df = df.drop(columns=cols_to_drop)
print(f"Remaining shape: {df.shape}")
print(df.columns.tolist())

In [ ]:
cols_id = ['route_name', 'origin', 'destination', 'route_name_simple']
df = df.drop(columns=cols_id)
print(f"Shape after dropping identifiers: {df.shape}")
print(df.columns.tolist())

In [ ]:
from sklearn.preprocessing import LabelEncoder

cat_cols = ['service_type', 'distance_category', 'operator',
            'origin_country', 'destination_country']

le = LabelEncoder()
for col in cat_cols:
    df[col + '_encoded'] = le.fit_transform(df[col].astype(str))
    print(f"{col:25s} → {col}_encoded  ({df[col].nunique()} unique values)")

print(f"\nShape after encoding: {df.shape}")

In [ ]:
df['is_domestic'] = (df['origin_country'] == df['destination_country']).astype(int)
df['distance_per_capacity'] = df['distance_km'] / df['capacity']
df['passengers_per_km'] = df['passengers_estimated'] / df['distance_km']

print("New features created:")
print(df[['is_domestic', 'distance_per_capacity', 'passengers_per_km']].describe())
print(f"\nShape after feature engineering: {df.shape}")

In [ ]:
cols_to_drop_str = ['service_type', 'distance_category', 'operator',
                    'origin_country', 'destination_country']

df_model = df.drop(columns=cols_to_drop_str)

X = df_model.drop(columns=['is_underserved'])
y = df_model['is_underserved']

print(f"Features (X) shape : {X.shape}")
print(f"Target  (y) shape  : {y.shape}")
print(f"\nFeature columns:\n{X.columns.tolist()}")
print(f"\nClass balance:\n{y.value_counts(normalize=True).round(4)}")

In [ ]:
from sklearn.model_selection import train_test_split

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp
)

print(f"Train      : {X_train.shape[0]} rows ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Validation : {X_val.shape[0]} rows ({X_val.shape[0]/len(X)*100:.1f}%)")
print(f"Test       : {X_test.shape[0]} rows ({X_test.shape[0]/len(X)*100:.1f}%)")
print(f"\nClass balance in train:\n{y_train.value_counts(normalize=True).round(4)}")
print(f"\nClass balance in test:\n{y_test.value_counts(normalize=True).round(4)}")

In [ ]:
import os
os.makedirs('../data/features', exist_ok=True)

df_model.to_csv('../data/features/trains_features.csv', index=False)
print(f"Saved to data/features/trains_features.csv")
print(f"Final shape: {df_model.shape}")
print(f"Columns: {df_model.columns.tolist()}")

In [ ]:
# Étape 0 — Vérification de la répartition is_underserved par type de service
# ObRail veut comparer trains de jour vs trains de nuit.
# On vérifie si les routes sous-desservies sont plus fréquentes le jour ou la nuit.

print("Taux de sous-desserte par type de service :")
print(df.groupby('service_type')['is_underserved'].value_counts(normalize=True).mul(100).round(1))

print("\nNombre de routes sous-desservies par type :")
print(df.groupby(['service_type', 'is_underserved']).size().unstack())

In [ ]:
# Est-ce que is_underserved est un seuil sur service_ratio ?
print(df.groupby('is_underserved')[['service_ratio', 'load_factor', 'capacity', 'passengers_estimated']].describe())

In [ ]:
print("Répartition is_cross_border :")
print(df['is_cross_border'].value_counts())
print(f"\nPourcentage : {df['is_cross_border'].mean()*100:.1f}%")

print("\nTaux de sous-desserte selon is_cross_border :")
print(df.groupby('is_cross_border')['is_underserved'].mean().mul(100).round(1))

In [ ]:
print("Valeurs uniques de capacity :")
print(df['capacity'].value_counts())

In [ ]:
import os
for f in os.listdir('../data/raw/'):
    print(f)

In [ ]:
for f in os.listdir('../data/processed/'):
    print(f)

In [ ]:
df_raw = pd.read_csv('../data/raw/all_routes_cleaned.csv')
print(df_raw.shape)
print(df_raw.columns.tolist())
print(df_raw.head(3))

In [ ]:
df_cities = pd.read_csv('../data/raw/cities15000.txt', sep='\t', header=None)
print(df_cities.shape)
print(df_cities.head(3))

In [ ]:
df_env = pd.read_csv('../data/raw/environmental_impact.csv')
print(df_env.shape)
print(df_env.columns.tolist())
print(df_env.head(3))

In [ ]:
import pandas as pd

# OpenFlights routes - actual flight routes data
df_flights = pd.read_csv('../data/external/openflights_routes.csv')
print("=== openflights_routes ===")
print(df_flights.shape)
print(df_flights.columns.tolist())
print(df_flights.head(3))

In [ ]:
# OpenFlights airports - airport info with coordinates
df_airports = pd.read_csv('../data/external/openflights_airports.csv')
print("=== openflights_airports ===")
print(df_airports.shape)
print(df_airports.columns.tolist())
print(df_airports.head(3))

In [ ]:
# Eurostat cities
df_euro_cities = pd.read_csv('../data/external/eurostat_cities_raw.csv')
print("=== eurostat_cities_raw ===")
print(df_euro_cities.shape)
print(df_euro_cities.columns.tolist())
print(df_euro_cities.head(3))

In [ ]:
Trip count distribution:
count    1366.000000
mean        4.438507
std         4.113975
min         1.000000
25%         2.000000
50%         3.000000
75%         6.000000
max        37.000000
Name: trip_count, dtype: float64

Sample:
  route_long_name  trip_count
0        ICE 1672           7
1        ICE 1579           4
2         IC 2375           9
3        ICE 1574           4
4        ICE 1572           7
5        ICE 1570           5
6        ICE 1571           2
7        ICE 1573           4
8        ICE 1674           7
9        ICE 1575           8

In [ ]:
print("\nRoute type 2, non-Swiss, non-DK sample:")
sample = df[
    (df['route_type'] == 2) & 
    (~df['country'].isin(['CH'])) &
    (df['route_name'].str.contains('IC|IR|TGV|RE|RB|TER|ICE|EC|EN|NJ|DB|SNCF', case=False, na=False))
]['route_name'].sample(20).tolist()
print(sample)